# Computational Ranking of Surface Micro-Texture Geometries for Aerodynamic Drag Reduction

**Kanav Thonda, Rouse High School (Class of 2028)**

Reduced-order semi-empirical study of 67 surface micro-texture geometries (riblets, dimples,
shark-skin denticles, hybrids) across 5 flow speeds on 2 test bodies (flat plate + sphere).

Pipeline: physics model -> validation against published benchmarks -> 670-row dataset ->
9 analyses -> 8 figures -> paper / web interface / handoff.

All modules live in `/workspace/` and are imported here so the notebook is the reproducible record.

In [15]:
import sys, os, subprocess, importlib
sys.path.insert(0, "/workspace")
os.chdir("/workspace")

import numpy as np
import pandas as pd
import texture_model as tm

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

print("numpy", np.__version__, "| pandas", pd.__version__)
print("Fluid: rho=%.3f kg/m3, mu=%.4e Pa.s, nu=%.4e m2/s" % (tm.RHO, tm.MU, tm.NU))
print("Plate L=%.2f m | Sphere D=%.4f m | Speeds %s m/s" % (tm.L_PLATE, tm.D_SPHERE, tm.SPEEDS))
print("Re_L  :", [f"{u*tm.L_PLATE/tm.NU:.3e}" for u in tm.SPEEDS])
print("Re_D  :", [f"{u*tm.D_SPHERE/tm.NU:.3e}" for u in tm.SPEEDS])

numpy 2.1.0 | pandas 2.3.1
Fluid: rho=1.184 kg/m3, mu=1.8490e-05 Pa.s, nu=1.5617e-05 m2/s
Plate L=0.10 m | Sphere D=0.0427 m | Speeds [1.0, 5.0, 10.0, 20.0, 50.0] m/s
Re_L  : ['6.403e+03', '3.202e+04', '6.403e+04', '1.281e+05', '3.202e+05']
Re_D  : ['2.734e+03', '1.367e+04', '2.734e+04', '5.469e+04', '1.367e+05']


## 1. Model validation against published benchmarks

Benchmarks are tagged by **evidential status**, because "13/13 passed" is misleading if the model
was fitted to those same numbers:

- **IMPLEMENTATION** - reproduces a published correlation; tests the code, not the physics.
- **CALIBRATED** - a model constant was set to hit this value; *not* independent evidence.
- **EMERGENT** - the model was never fitted to this target; the only genuine predictive tests.

In [16]:
r = subprocess.run([sys.executable, "validate_model.py"], capture_output=True, text=True, cwd="/workspace")
print(r.stdout[-2500:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-2500:])

val = pd.read_csv("/workspace/validation_benchmarks.csv")
print("\nBenchmarks by evidential status:")
print(val.groupby("status").passed.agg(["sum", "count"]))
print("\nEMERGENT (independent) tests:")
print(val[val.status == "EMERGENT"][["benchmark", "published", "predicted", "passed"]].to_string(index=False))

NT IMPLEMENTATION         0.0045     0.00447     -      -0.65    True
         Smooth plate C_F at Re_L=1e7                              Schlichting, Boundary-Layer Theory POINT IMPLEMENTATION          0.003     0.00300     -       0.12    True
         Smooth sphere Cd at Re_D=1e5                                  Clift & Gauvin; Achenbach 1972 POINT IMPLEMENTATION            0.5     0.49175     -      -1.65    True
                 Blade riblet peak DR                                Bechert et al. 1997 (JFM 338:59) POINT     CALIBRATED            9.9     9.89998     %      -0.00    True
              Blade riblet optimal s+                                 Bechert et al. 1997; Walsh 1983  BAND       EMERGENT       13 to 20    15.52572     -        NaN    True
       Blade riblet DR->0 crossing s+                          Bechert 1997: drag rise beyond s+ ~ 30  BAND       EMERGENT       25 to 40    31.05138     -        NaN    True
                     V-groove peak DR                  

## 2. Geometry catalogue and dataset generation

67 geometries x 5 speeds x 2 bodies = 670 rows. The riblet spacing x height grid and the dimple
diameter x depth grid are deliberately *full* factorial so the Graph 3 and Graph 5 heatmaps have
zero empty cells.

In [17]:
r = subprocess.run([sys.executable, "build_dataset.py"], capture_output=True, text=True, cwd="/workspace")
print(r.stdout[-2500:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-2500:])

df = pd.read_csv("/workspace/dataset.csv")

# --- acceptance checks from the approved plan ---
checks = {
    "670 rows": len(df) == 670,
    "67 geometries": df.geometry_id.nunique() == 67,
    "class counts 21/20/15/10/1": df.groupby("geometry_class").geometry_id.nunique().to_dict()
        == {"baseline": 1, "dimple": 20, "hybrid": 10, "riblet": 21, "shark": 15},
    "33 columns": len(df.columns) == 33,
    "no NaN in DR_net_pct": df.DR_net_pct.isna().sum() == 0,
    "baseline DR exactly 0": np.allclose(df[df.geometry_class == "baseline"].DR_net_pct, 0.0),
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "dataset acceptance checks failed"

print(f"\nDR_net_pct range: plate {df[df.body=='plate'].DR_net_pct.min():.2f} to "
      f"{df[df.body=='plate'].DR_net_pct.max():.2f} % | "
      f"sphere {df[df.body=='sphere'].DR_net_pct.min():.2f} to "
      f"{df[df.body=='sphere'].DR_net_pct.max():.2f} %")

rows=670  geometries=67  columns=33
geometry_class
baseline     1
dimple      20
hybrid      10
riblet      21
shark       15

Best 8 on the PLATE at 10 m/s:
     geometry_id geometry_class    s_plus  DR_net_pct  DR_uncertainty_pp
   A-V-s500-h250         riblet 20.152368    5.344308                1.5
A-BLAD-s200-h100         riblet  8.060947    5.140251                1.5
   A-V-s200-h250         riblet  8.060947    2.839667                1.5
 A-BLAD-s100-h50         riblet  4.030474    2.570125                1.5
   A-V-s200-h100         riblet  8.060947    2.137723                1.5
         D-HY-03         hybrid  4.030474    2.041243                4.0
   A-V-s100-h100         riblet  4.030474    1.495152                1.5
         D-HY-05         hybrid  8.060947    1.286046                4.0

Best 8 on the SPHERE at 50 m/s:
     geometry_id geometry_class  Cd_total  DR_net_pct  DR_uncertainty_pp
 B-HEX-d2.0-r0.1         dimple  0.231553   52.768469                4.1
 B-HEX

## 3. The nine analyses

All nine live in `analyses.py` and are driven by one `run_all()` call, so the paper, the figures
and the web interface all quote the same numbers. Rankings carry a **statistical tie** count:
the number of other geometries whose uncertainty interval overlaps the winner's. With a calibrated
reduced-order model that is the honest resolution limit.

In [18]:
import build_dataset as bd
import analyses as an
importlib.reload(bd); importlib.reload(an)

catalogue = bd.build_catalogue()
R = an.run_all(df, catalogue)
print("catalogue entries:", len(catalogue), "| analyses:", list(R))

catalogue entries: 67 | analyses: ['a1_ranking', 'a2_regime', 'a3_riblet', 'a4_dimple', 'a5_shark', 'a6_hybrid', 'a7_speed', 'a8_pareto', 'a9_stats']


### A1-A2 | Overall ranking and the best geometry in each flow regime

In [19]:
print("Top 10 on the flat plate at 10 m/s:")
print(R["a1_ranking"]["plate_10"].head(10).round(3).to_string(index=False))

print("\nTop 10 on the sphere at 50 m/s:")
print(R["a1_ranking"]["sphere_50"].head(10).round(3).to_string(index=False))

print("\nA2 | Best geometry per flow regime (ties = overlapping uncertainty intervals):")
print(R["a2_regime"].round(3).to_string(index=False))

Top 10 on the flat plate at 10 m/s:
 rank      geometry_id geometry_class           shape  DR_net_pct  DR_uncertainty_pp model_confidence
    1    A-V-s500-h250         riblet        v-groove       5.344                1.5             high
    2 A-BLAD-s200-h100         riblet           blade       5.140                1.5             high
    3    A-V-s200-h250         riblet        v-groove       2.840                1.5             high
    4  A-BLAD-s100-h50         riblet           blade       2.570                1.5             high
    5    A-V-s200-h100         riblet        v-groove       2.138                1.5             high
    6          D-HY-03         hybrid    blade+dimple       2.041                4.0              low
    7    A-V-s100-h100         riblet        v-groove       1.495                1.5             high
    8          D-HY-05         hybrid v-groove+dimple       1.286                4.0              low
    9  A-SCAL-s100-h50         riblet       sc

### A3 | Riblet optimum: discrete catalogue vs continuous sweep

The 21 catalogue riblets only *sample* the performance curve. Blade riblets in particular exist
only at s = 100 and 200 um, so the discrete grid can miss their true optimum. The continuous
per-shape sweep locates the real optimum and is what the paper quotes.

In [20]:
print("Discrete winner at each speed:")
print(R["a3_riblet"]["discrete"].round(3).to_string(index=False))

print("\nContinuous per-shape optimum (h/s held at its optimal value):")
print(R["a3_riblet"]["continuous"].round(3).to_string(index=False))

b = R["a3_riblet"]["band"]
print(f"\nRiblet rows landing inside s+ = 10-20 : {b['n_rows_in_band']}")
print(f"  mean DR inside band  : {b['mean_DR_in_band']:.3f} %")
print(f"  mean DR outside band : {b['mean_DR_out_of_band']:.3f} %")
print(f"  best DR inside band  : {b['best_DR_in_band']:.3f} %")
print(f"  -> being in the band is worth {b['mean_DR_in_band'] - b['mean_DR_out_of_band']:.2f} pp on average")
print("\nNote: l_g+ collapses to ~10.7 for all four shapes while s+ spans 15.5-18.1,")
print("reproducing the Garcia-Mayoral & Jimenez result that l_g+ (not s+) is the governing scale.")

Discrete winner at each speed:
 U_inf_mps      geometry_id    shape  spacing_um  height_um  h_over_s  s_plus  lg_plus  DR_net_pct
       1.0    A-V-s500-h250 v-groove       500.0      250.0       0.5   2.723    1.361       0.722
       5.0    A-V-s500-h250 v-groove       500.0      250.0       0.5  10.953    5.476       2.905
      10.0    A-V-s500-h250 v-groove       500.0      250.0       0.5  20.152   10.076       5.344
      20.0 A-BLAD-s200-h100    blade       200.0      100.0       0.5  14.907   10.274       9.506
      50.0  A-BLAD-s100-h50    blade       100.0       50.0       0.5  16.915   11.658       9.471

Continuous per-shape optimum (h/s held at its optimal value):
    shape  h_over_s_opt  DR_peak_pct  s_plus_opt  lg_plus_opt  s_plus_zero_crossing
    blade           0.5        9.899      15.541       10.711                31.081
scalloped           0.7        6.499      16.508       10.698                33.075
 v-groove           0.7        6.200      18.088       10.70

### A4-A5 | Dimple geometry effects and shark-denticle scaling

In [21]:
d4 = R["a4_dimple"]
print("A4 | Plate mean DR (%) by depth ratio d/D x speed  [hexagonal, 40% coverage]:")
print(d4["by_depth"].to_string())
print("\nA4 | Plate mean DR (%) by dimple diameter x speed:")
print(d4["by_diameter"].to_string())
print("\nA4 | Best plate dimple at each speed:")
print(d4["best_plate_per_speed"].round(3).to_string(index=False))
print(f"\nA4 | Plate dimple mean over all rows: {d4['plate_mean']:.3f} %  "
      f"| sphere dimple max: {d4['sphere_max']:.3f} %")

print("\nA4 | Sphere drag-crisis onset (first speed exceeding 20% DR):")
ons = d4["sphere_onset"]
print(f"  {ons.onset_speed_mps.notna().sum()} of {len(ons)} dimple geometries eventually trip the crisis")
print(f"  earliest onset at {ons.onset_speed_mps.min():g} m/s:",
      ", ".join(ons[ons.onset_speed_mps == ons.onset_speed_mps.min()].geometry_id))

A4 | Plate mean DR (%) by depth ratio d/D x speed  [hexagonal, 40% coverage]:
U_inf_mps      1.0     5.0     10.0    20.0    50.0
depth_ratio                                        
0.05         -0.556  -0.412  -0.200   0.073   0.361
0.10         -2.223  -2.037  -1.898  -1.784  -1.733
0.20         -8.950  -8.911  -8.897  -8.891  -8.903
0.30        -20.157 -20.150 -20.149 -20.149 -20.153

A4 | Plate mean DR (%) by dimple diameter x speed:
U_inf_mps     1.0    5.0    10.0   20.0   50.0
diameter_mm                                   
0.5         -7.980 -7.977 -7.964 -7.919 -7.752
1.0         -7.980 -7.961 -7.909 -7.787 -7.545
2.0         -7.977 -7.897 -7.766 -7.581 -7.457
5.0         -7.949 -7.675 -7.504 -7.464 -7.671

A4 | Best plate dimple at each speed:
 U_inf_mps      geometry_id  diameter_mm  depth_ratio  DR_net_pct
       1.0 B-HEX-d5.0-r0.05          5.0         0.05      -0.543
       5.0 B-HEX-d5.0-r0.05          5.0         0.05      -0.047
      10.0 B-HEX-d5.0-r0.05          5.

In [22]:
d5 = R["a5_shark"]
print("A5 | Plate mean DR (%) by denticle scale x speed:")
print(d5["by_scale"].to_string())
print("\nA5 | Plate mean DR (%) by denticle overlap x speed:")
print(d5["by_overlap"].to_string())
print("\nA5 | Best denticle at each speed:")
print(d5["best_per_speed"].round(3).to_string(index=False))
print(f"\nA5 | Shark-skin peak across the whole study: {d5['peak']:.3f} %")
print("\nA5 | Denticles vs equivalent ideal riblets:")
print(d5["vs_riblet"].round(3).to_string(index=False))

A5 | Plate mean DR (%) by denticle scale x speed:
U_inf_mps           1.0    5.0    10.0   20.0   50.0
denticle_scale_um                                   
50.0              -0.626 -0.585 -0.538 -0.452 -0.214
100.0             -0.614 -0.535 -0.446 -0.282  0.173
200.0             -0.545 -0.378 -0.191  0.156  1.115
500.0             -0.503 -0.088  0.376  1.239  1.622

A5 | Plate mean DR (%) by denticle overlap x speed:
U_inf_mps     1.0    5.0    10.0   20.0   50.0
overlap_pct                                   
0.0         -0.748 -0.591 -0.416 -0.090  0.366
20.0        -0.593 -0.451 -0.292  0.003  0.486
40.0        -0.417 -0.226 -0.013  0.383  1.034

A5 | Best denticle at each speed:
 U_inf_mps   geometry_id  denticle_scale_um  overlap_pct  s_plus  DR_net_pct
       1.0 C-SK-s500-o40              500.0         40.0   0.908      -0.328
       5.0 C-SK-s500-o40              500.0         40.0   3.651       0.133
      10.0 C-SK-s500-o40              500.0         40.0   6.717       0.649
 

### A6 | Do hybrids beat their constituents?

Each hybrid is compared against the pure riblet and the pure dimple it is built from, both
re-evaluated at identical conditions. This is the cleanest test of the sub-additive assumption.

In [23]:
h = R["a6_hybrid"]
s6 = h["summary"]
print(f"Hybrid x speed cases evaluated      : {s6['n_hybrid_speed_cases']}")
print(f"Cases beating best constituent      : {s6['n_beating_best_constituent']} "
      f"({s6['frac_beating']*100:.0f} %)")
print(f"Mean interference penalty           : {s6['mean_interference_pp']:.3f} pp")
print(f"Mean hybrid DR                      : {s6['mean_hybrid']:.3f} %")
print(f"Mean best-constituent DR            : {s6['mean_best_constituent']:.3f} %")
print(f"\n-> Hybridisation costs {s6['mean_best_constituent'] - s6['mean_hybrid']:.2f} pp "
      "relative to simply using the better single texture.")
print("\nPer-hybrid detail at 10 m/s:")
print(h["table"][h["table"].U_inf_mps == 10]
      [["geometry_id", "riblet_alone", "dimple_alone", "linear_sum", "hybrid",
        "interference_pp", "beats_best_constituent"]].round(3).to_string(index=False))

Hybrid x speed cases evaluated      : 50
Cases beating best constituent      : 0 (0 %)
Mean interference penalty           : -0.605 pp
Mean hybrid DR                      : -0.899 %
Mean best-constituent DR            : 2.024 %

-> Hybridisation costs 2.92 pp relative to simply using the better single texture.

Per-hybrid detail at 10 m/s:
geometry_id  riblet_alone  dimple_alone  linear_sum  hybrid  interference_pp  beats_best_constituent
    D-HY-01         1.069        -0.110       0.959   0.540           -0.419                   False
    D-HY-02         1.069        -0.263       0.806  -0.002           -0.808                   False
    D-HY-03         2.570        -0.110       2.461   2.041           -0.419                   False
    D-HY-04         2.570        -2.088       0.482  -0.224           -0.707                   False
    D-HY-05         2.138         0.468       2.605   1.286           -1.319                   False
    D-HY-06         5.140        -2.334       2.806 

### A7-A9 | Speed sensitivity, Pareto fronts, and per-class statistics

In [24]:
s7 = R["a7_speed"]
print("A7 | Top-5 plate geometries across all speeds (DR %, columns = m/s):")
print(s7["plate"].to_string())
print("\nA7 | Top-5 sphere geometries across all speeds:")
print(s7["sphere"].to_string())
print("\nA7 | Class mean DR (%) by body and speed:")
print(s7["class_means"].to_string())

A7 | Top-5 plate geometries across all speeds (DR %, columns = m/s):
U_inf_mps           1.0    5.0   10.0   20.0    50.0  range_pp  sign_flips
geometry_id                                                               
A-V-s500-h250     0.722  2.905  5.344  1.829 -17.506    22.850           1
A-BLAD-s200-h100  0.694  2.794  5.140  9.506  -2.363    11.869           1
A-V-s200-h250     0.384  1.543  2.840  4.525  -3.305     7.830           1
A-BLAD-s100-h50   0.347  1.397  2.570  4.753   9.471     9.124           0
A-V-s200-h100     0.289  1.162  2.138  3.953   2.875     3.664           0

A7 | Top-5 sphere geometries across all speeds:
U_inf_mps           1.0    5.0   10.0   20.0    50.0  range_pp  sign_flips
geometry_id                                                               
B-HEX-d2.0-r0.1  -3.044 -2.716 -2.385  4.546  52.768    55.812           2
B-HEX-d1.0-r0.2  -3.044 -2.722 -2.398  4.528  52.756    55.800           2
D-HY-05          -3.433 -3.036 -2.611  6.267  52.593    5

In [25]:
p8 = R["a8_pareto"]
print("A8 | Pareto front, plate @ 10 m/s, drag reduction vs minimum feature size:")
print(p8["plate"]["feature_size"].round(3).to_string(index=False))
print("\nA8 | Pareto front, plate @ 10 m/s, drag reduction vs manufacturability index:")
print(p8["plate"]["manufacturability"].round(3).to_string(index=False))
print("\nA8 | Best geometry at each manufacturability tier (plate @ 10 m/s):")
print(p8["plate_by_tier"].round(3).to_string(index=False))

A8 | Pareto front, plate @ 10 m/s, drag reduction vs minimum feature size:
    geometry_id geometry_class  min_feature_um  DR_net_pct  DR_uncertainty_pp
  A-V-s500-h250         riblet           250.0       5.344                1.5
B-HEX-d5.0-r0.1         dimple           500.0      -1.498                2.5
B-HEX-d5.0-r0.2         dimple          1000.0      -8.886                2.5
B-HEX-d5.0-r0.3         dimple          1500.0     -20.152                2.5

A8 | Pareto front, plate @ 10 m/s, drag reduction vs manufacturability index:
  geometry_id geometry_class  manufacturability_index  DR_net_pct  DR_uncertainty_pp
A-V-s500-h250         riblet                      1.0       5.344                1.5

A8 | Best geometry at each manufacturability tier (plate @ 10 m/s):
 tier      geometry_id geometry_class  min_feature_um  manufacturability_index  DR_net_pct
    1    A-V-s500-h250         riblet           250.0                      1.0       5.344
    2 B-HEX-d5.0-r0.05         dimp

In [26]:
st = R["a9_stats"]
for body in ["plate", "sphere"]:
    print(f"A9 | {body.upper()} - distribution of net drag reduction (%) by class, all speeds:")
    print(st[body].to_string())
    print()

A9 | PLATE - distribution of net drag reduction (%) by class, all speeds:
                n_rows  n_geometries   mean     sd  median     min    max  frac_positive     best_geometry   worst_geometry
geometry_class                                                                                                             
riblet             105            21  1.091  2.639   0.595 -17.506  9.506          0.962  A-BLAD-s200-h100    A-V-s500-h250
dimple             100            20 -6.253  7.713  -2.048 -20.160  1.354          0.140        B-HEXA-c80  B-HEX-d0.5-r0.3
shark               75            15 -0.096  0.647  -0.321  -0.788  2.033          0.280     C-SK-s500-o40      C-SK-s50-o0
hybrid              50            10 -0.899  5.120  -0.113 -19.583  9.135          0.480           D-HY-03          D-HY-09
baseline             5             1  0.000  0.000   0.000   0.000  0.000          0.000          E-SMOOTH         E-SMOOTH

A9 | SPHERE - distribution of net drag reduction (%) by c

## 4. Figures

Eight figures, each saved as **SVG** (editable text) and **PNG** (for the rendering check).
Graph 6 is built from a numerically solved Blasius profile and Spalding's law of the wall, so the
laminar/turbulent contrast is quantitatively correct and the wall geometry is drawn to scale in
micrometres rather than sketched.

In [27]:
import shutil
import figures as fg
importlib.reload(fg)

os.makedirs("/mnt/results", exist_ok=True)

# graph 6 needs the best riblet and best dimple on the plate at 10 m/s
ref = df[(df.body == "plate") & (df.U_inf_mps == 10)]
best_riblet = ref[ref.geometry_class == "riblet"].nlargest(1, "DR_net_pct").iloc[0]
best_dimple = ref[ref.geometry_class == "dimple"].nlargest(1, "DR_net_pct").iloc[0]
print(f"Graph 6 references: riblet {best_riblet.geometry_id} ({best_riblet.DR_net_pct:.2f}%), "
      f"dimple {best_dimple.geometry_id} ({best_dimple.DR_net_pct:.2f}%)")

made = [fg.graph1(df), fg.graph2(df), fg.graph3(df), fg.graph4(df), fg.graph5(df),
        fg.graph6(df, best_riblet, best_dimple), fg.graph7(df), fg.graph8(df)]

# export the tabular deliverables alongside the figures
df.to_csv("/mnt/results/dataset.csv", index=False)
shutil.copy("/workspace/validation_benchmarks.csv", "/mnt/results/validation_benchmarks.csv")

for f in sorted(os.listdir("/mnt/results")):
    p = f"/mnt/results/{f}"
    if os.path.isfile(p):
        print(f"  {os.path.getsize(p):>9,} B  {f}")

Graph 6 references: riblet A-V-s500-h250 (5.34%), dimple B-HEX-d5.0-r0.05 (0.52%)


    147,118 B  dataset.csv
     50,795 B  graph1_top20_drag_reduction_bar.png
     28,525 B  graph1_top20_drag_reduction_bar.svg
    140,828 B  graph2_drag_vs_flowspeed_top_geometries.png
     64,281 B  graph2_drag_vs_flowspeed_top_geometries.svg
     26,103 B  graph3_heatmap_riblet_spacing_height.png
     19,620 B  graph3_heatmap_riblet_spacing_height.svg
     76,806 B  graph4_scatter_splus_vs_drag.png
     49,156 B  graph4_scatter_splus_vs_drag.svg
     30,630 B  graph5_heatmap_dimple_diameter_depth.png
     20,960 B  graph5_heatmap_dimple_diameter_depth.svg
    205,191 B  graph6_flow_visualization_streamlines.png
    520,123 B  graph6_flow_visualization_streamlines.svg
     81,180 B  graph7_pareto_drag_vs_feature_size.png
     57,186 B  graph7_pareto_drag_vs_feature_size.svg
     79,301 B  graph8_boxplots_geometry_classes.png
    148,659 B  graph8_boxplots_geometry_classes.svg
    180,692 B  results_summary.json
      1,505 B  validation_benchmarks.csv


In [28]:
import importlib
import summary as sm
importlib.reload(sm)

S = sm.main()
print()
print("headline class means, plate :", S["headline"]["plate_class_mean"])
print("headline class means, sphere:", S["headline"]["sphere_class_mean"])
print("riblet is the only class with a positive plate mean:",
      S["headline"]["riblet_only_positive_plate_mean"])
print("sphere best / plate best   :", S["derived"]["sphere_vs_plate_ratio"], "x")
print("hybrids beating their best constituent:",
      S["a6_hybrid"]["summary"]["n_beating_best_constituent"], "/",
      S["a6_hybrid"]["summary"]["n_hybrid_speed_cases"])

wrote /mnt/results/results_summary.json
  top-level keys: ['meta', 'validation', 'headline', 'a1_ranking', 'a2_regime', 'a3_riblet', 'a4_dimple', 'a5_shark', 'a6_hybrid', 'a7_speed', 'a8_pareto', 'a9_stats', 'derived']
  best plate  : {'geometry_id': 'A-BLAD-s200-h100', 'geometry_class': 'riblet', 'shape': 'blade', 'U': 20.0, 'DR': 9.506, 'unc': 1.5, 's_plus': 14.907}
  best sphere : {'geometry_id': 'B-HEX-d2.0-r0.1', 'geometry_class': 'dimple', 'U': 50.0, 'DR': 52.768, 'unc': 4.1, 'Cd_textured': 0.231553, 'Cd_smooth': 0.490251}
  validation  : 13/13 passed, statuses {'CALIBRATED': 7, 'IMPLEMENTATION': 3, 'EMERGENT': 3}

headline class means, plate : {'riblet': 1.091, 'dimple': -6.253, 'shark': -0.096, 'hybrid': -0.899, 'baseline': 0.0}
headline class means, sphere: {'riblet': 1.718, 'dimple': 7.955, 'shark': 2.248, 'hybrid': 7.014, 'baseline': 0.0}
riblet is the only class with a positive plate mean: True
sphere best / plate best   : 5.55 x
hybrids beating their best constituent: 0 / 

In [29]:
import importlib
import subprocess

import refs, paper, paper2
for m in (refs, paper2, paper):
    importlib.reload(m)

NUM, BIB = refs.build()
print(f"bibliography: {len(BIB)} real, DOI-verified references")

print(subprocess.run(["python", "/workspace/paper.py"], capture_output=True,
                     text=True, cwd="/workspace").stdout)

text = open("/mnt/results/research_paper.md").read()
secs = [l for l in text.splitlines() if l.startswith("## ")]
body = text.split("## 8. References")[0]
print("sections:", len(secs))
for s in secs:
    print("  ", s[3:])
checks = {
    "11 sections": len(secs) == 11,
    ">= 5000 words (excl. references/appendices)": len(body.split()) >= 5000,
    ">= 18 citations": len(BIB) >= 18,
    "all 8 figures referenced": all(f"graph{i}" in text for i in range(1, 9)),
    "no unresolved interpolation": "{M[" not in text and "{A" not in text,
}
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values())

bibliography: 41 real, DOI-verified references


wrote /mnt/results/research_paper.md
blocks          : 31
total words     : 32519
words pre-refs  : 13618
prose words     : 10908
characters      : 173524
markdown tables : 22

sections: 11
   1. Abstract
   2. Introduction
   3. Theoretical Background
   4. Methodology
   5. Results
   6. Discussion
   7. Conclusion
   8. References
   Appendix A. Full drag coefficient table
   Appendix B. Boundary layer model equations and assumptions
   Appendix C. Geometry parameter definitions and diagrams
PASS  11 sections
PASS  >= 5000 words (excl. references/appendices)
PASS  >= 18 citations
PASS  all 8 figures referenced
PASS  no unresolved interpolation


## 7. Handoff document generation

`handoff.py` reads `results_summary.json` and `dataset.csv` and emits the 8-section handoff
document for the downstream local agent. Every quantity is read programmatically, so the
document stays consistent with the dataset if the physics model is ever changed.

In [30]:
cd /workspace && python handoff.py

wrote /mnt/results/handoff.md


  characters      : 57095


  words           : 8983


  numbered sects  : ['1', '2', '3', '4', '5', '6', '7', '8']


  markdown tables : 14


  next-step items : 12


  open questions  : 10


  experiments     : 4


  unresolved f-str: none


  ALL CHECKS PASS


In [31]:
cd /workspace && python webbuild.py

wrote /mnt/results/index.html


  size            : 1.30 MB (1362596 bytes)


  sections present: 11/11


  nav links       : 11


  inline <svg>    : 8 (8 figures expected)


...

  embedded handoff: 57099 chars


  download hooks  : 6


  inline JS #0      : 3136 chars, 0 code backticks


  inline JS #1      : 25010 chars, 92 code backticks


  ALL CHECKS PASS


In [32]:
cd /workspace && python webtest.py 2>/dev/null | python -c "
import sys, json
s = sys.stdin.read(); r = json.loads(s[:s.rindex('}')+1])
for k, v in r.items():
    if k == 'explorer':
        print('explorer:')
        for kk, vv in v.items():
            print(f'   {kk:26s} DR={vv[\"dr\"]:>8s}  xsec_paths={vv[\"xsec_paths\"]}')
    elif k == 'downloads':
        print('downloads:')
        for kk, vv in v.items():
            print(f'   {kk:22s} {vv[\"bytes\"]:>7d} B  name_ok={vv[\"match\"]}')
    else:
        print(f'{k:26s} {str(v)[:118]}')
assert not r['console_errors'], r['console_errors']
print()
print('BROWSER TEST: all assertions passed, 0 console errors')
"

title                      Surface Micro-Texture Drag Reduction — Kanav Thonda


sections                   ['hero', 'abstract', 'findings', 'explorer', 'compare', 'figures', 'flow', 'rankings', 'paper', 'downloads', 'handoff'


nav_links                  11


chartjs_loaded             True


rows_parsed                670


...

mobile_swipehint_visible   True


mobile_table_scrollable    True


console_errors             []


console_warnings           []


BROWSER TEST: all assertions passed, 0 console errors


## 8. Interactive web interface

`webbuild.py` assembles a single self-contained `index.html`: the 670-row dataset, the
full manuscript and the handoff document are embedded verbatim as `<script type="text/plain">`
payloads, the eight matplotlib SVGs are inlined with namespaced ids, and ~25 kB of vanilla JS
plus Chart.js drives the drag explorer, the multi-geometry comparison tool, a parametric
to-scale cross-section renderer and the sortable 670-row ranking table.

Build-time acceptance checks cover section/nav counts, payload lengths, download hooks and
— after an unterminated template literal silently killed the whole app script earlier in this
session — an escape-aware balanced-backtick check on every inline `<script>` body.

`webtest.py` then drives the real page in headless Chromium (Playwright) and asserts the
rendered behaviour: parsed row/geometry counts, explorer readouts against `dataset.csv`,
cross-section path counts per texture class, comparison-tool selection persistence across
search filtering, table sorting/searching/tinting, blob download filenames and byte sizes,
mobile layout (no horizontal scroll at 390 px), and an empty JS console.

## 6. Manuscript generation

`refs.py` builds the AIAA bibliography from verified literature records; `paper.py` +
`paper2.py` interpolate every quantitative statement from `results_summary.json` and
`dataset.csv`, so no number in the manuscript is typed by hand.

## Single source of truth

Every headline number the manuscript and the web interface quote is exported once to
`results_summary.json`. Nothing downstream retypes a value by hand, so the dataset,
the figures, the paper and the website cannot drift apart.